# EXP-156: 유전자별 변이 효과 압축

- GitHub Issue: `#156`
- 공식 EXP-ID: `EXP-156`
- 브랜치: `issue-156-exp-gene-variant-effect-compression`
- 부모 실험: `EXP-094` (frozen Feature Spec v1)

이 노트북은 공식 실험 로직을 중복 구현하지 않습니다. 팀 공용 parser, Feature Factory, canonical CV runner와 EXP-156 config를 검증하고 공식 runner를 실행하는 안내 노트북입니다. 실제 결과의 단일 원본은 `reports/exp156_gene_variant_effect_compression/metrics.json`과 `EXPERIMENT_HISTORY.md`입니다.


## 1. 저장소 루트와 필수 파일 확인

노트북 위치와 관계없이 Git 저장소 루트를 자동으로 찾습니다. 원본 CSV는 읽기 전용 입력이며 Git에 추가하지 않습니다.


In [ ]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from scipy import sparse


def run_command(*arguments: str, check: bool = True) -> str:
    result = subprocess.run(
        list(arguments),
        check=check,
        capture_output=True,
        text=True,
    )
    return result.stdout.strip()


ROOT = Path(
    run_command("git", "rev-parse", "--show-toplevel")
).resolve()

CONFIG_PATH = ROOT / "configs" / "exp156_gene_variant_effect_compression.yaml"
RUNNER_PATH = ROOT / "scripts" / "run_exp156_gene_variant_effect_compression.py"
FEATURE_MODULE_PATH = ROOT / "src" / "open_cancer" / "compact_effect_features.py"
TEST_MODULE_PATH = ROOT / "tests" / "test_compact_effect_features.py"
TRAIN_PATH = ROOT / "data" / "raw" / "train.csv"
TEST_PATH = ROOT / "data" / "raw" / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "data" / "raw" / "sample_submission.csv"
SPLIT_PATH = ROOT / "data" / "splits" / "stratified_5fold_seed42.csv"

required_paths = (
    ROOT / "PROJECT_CONTEXT.md",
    ROOT / "EXPERIMENT_HISTORY.md",
    CONFIG_PATH,
    RUNNER_PATH,
    FEATURE_MODULE_PATH,
    TEST_MODULE_PATH,
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
    SPLIT_PATH,
)

missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError("필수 파일이 없습니다:\n" + "\n".join(missing))

print("Repository root:", ROOT)
print("필수 파일 확인 완료")


## 2. Issue·브랜치·실험 ID 계약 확인

공식 실행은 Issue 번호가 포함된 올바른 브랜치에서만 허용합니다.


In [ ]:
import sys

src_path = str(ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from open_cancer.experiment import resolve_experiment_context

context = resolve_experiment_context("experiment", cwd=ROOT)

assert context.branch == "issue-156-exp-gene-variant-effect-compression", context.branch
assert context.issue_number == 156, context.issue_number
assert context.experiment_id == "EXP-156", context.experiment_id

print("Branch:", context.branch)
print("Issue:", context.issue_number)
print("Experiment:", context.experiment_id)


## 3. Config 계약 확인

부모 Feature Spec, canonical split, 모델과 변경 범위가 EXP-156 가설과 일치하는지 확인합니다.


In [ ]:
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

assert config["run_mode"] == "experiment"
assert config["record_role"] == "official"
assert config["parent_experiment"] == "EXP-094"
assert config["feature_spec"]["name"] == "v1"
assert config["split"]["path"] == "data/splits/stratified_5fold_seed42.csv"
assert config["split"]["n_splits"] == 5
assert config["model"]["name"] == "xgboost"
assert config["training"]["balanced_sample_weight"] is True

display({
    "slug": config["slug"],
    "parent_experiment": config["parent_experiment"],
    "feature_spec": config["feature_spec"],
    "split": config["split"],
    "model": config["model"],
})


## 4. 데이터 계약과 canonical fold 확인

이 단계는 타깃을 이용해 특징을 선택하지 않습니다. ID, 행 수, 유전자 순서, 클래스와 fold 정렬만 검증합니다. test의 SUBCLASS는 존재하지 않으며 test 분포를 특징 선택에 사용하지 않습니다.


In [ ]:
from open_cancer.constants import (
    CLASS_LABELS,
    EXPECTED_GENE_COLUMNS,
    EXPECTED_TEST_ROWS,
    EXPECTED_TRAIN_ROWS,
)

train_header = pd.read_csv(TRAIN_PATH, nrows=0).columns.tolist()
test_header = pd.read_csv(TEST_PATH, nrows=0).columns.tolist()
train_meta = pd.read_csv(TRAIN_PATH, usecols=["ID", "SUBCLASS"], dtype=str)
test_meta = pd.read_csv(TEST_PATH, usecols=["ID"], dtype=str)
split = pd.read_csv(SPLIT_PATH, dtype={"ID": str, "fold": int})

train_genes = train_header[2:]
test_genes = test_header[1:]
assert len(train_meta) == EXPECTED_TRAIN_ROWS
assert len(test_meta) == EXPECTED_TEST_ROWS
assert len(train_genes) == EXPECTED_GENE_COLUMNS
assert train_genes == test_genes
assert set(train_meta["SUBCLASS"]) == set(CLASS_LABELS)

aligned = train_meta[["ID"]].merge(
    split, on="ID", how="left", validate="one_to_one", sort=False
)
assert aligned["ID"].equals(train_meta["ID"])
assert not aligned["fold"].isna().any()
assert set(aligned["fold"]) == set(range(5))

print("Train rows:", len(train_meta))
print("Test rows:", len(test_meta))
print("Genes:", len(train_genes))
print("Classes:", len(CLASS_LABELS))
print("Fold counts:", aligned["fold"].value_counts().sort_index().to_dict())


## 5. EXP-156 단위 테스트

공용 parser 사용, 압축값, 제거 열 범위와 공용 model runner 계약을 데이터 학습 전에 검사합니다.


In [ ]:
test_result = subprocess.run(
    [
        "uv",
        "run",
        "pytest",
        "tests/test_compact_effect_features.py",
        "tests/test_model_runner.py",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
test_result.check_returncode()


## 6. 학습 없이 Feature Spec materialize

Frozen v1을 생성한 뒤 유전자별 mutation-type indicator 5개를 압축 특징 4개로 대체합니다. 이 단계는 모델을 학습하거나 공식 점수를 만들지 않습니다. 결과 캐시는 `data/processed/`에 저장되며 Git에 올리지 않습니다.


In [ ]:
from open_cancer.compact_effect_features import materialize_compact_effect_ablation

NOTEBOOK_FEATURE_DIR = (
    ROOT / "data" / "processed" / "exp156_notebook_preflight_features"
)

feature_manifest = materialize_compact_effect_ablation(
    root=ROOT,
    name="v1",
    output_dir=NOTEBOOK_FEATURE_DIR,
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
)

display({
    "name": feature_manifest["name"],
    "parent_name": feature_manifest["parent_name"],
    "parent_experiment": feature_manifest["parent_experiment"],
    "feature_spec_sha256": feature_manifest["feature_spec_sha256"],
    "replacement": feature_manifest["replacement"],
    "train_shape": feature_manifest["train_shape"],
    "test_shape": feature_manifest["test_shape"],
    "train_qc": feature_manifest["train_qc"],
    "test_qc": feature_manifest["test_qc"],
})


In [ ]:
train_features = sparse.load_npz(
    NOTEBOOK_FEATURE_DIR / "train_features.npz"
).tocsr()
test_features = sparse.load_npz(
    NOTEBOOK_FEATURE_DIR / "test_features.npz"
).tocsr()
feature_names = json.loads(
    (NOTEBOOK_FEATURE_DIR / "feature_names.json").read_text(encoding="utf-8")
)

assert train_features.shape[0] == EXPECTED_TRAIN_ROWS
assert test_features.shape[0] == EXPECTED_TEST_ROWS
assert train_features.shape[1] == test_features.shape[1] == len(feature_names)
assert np.isfinite(train_features.data).all()
assert np.isfinite(test_features.data).all()
assert len(feature_names) == len(set(feature_names))

print("Train feature shape:", train_features.shape)
print("Test feature shape:", test_features.shape)
print("Train nnz:", train_features.nnz)
print("Test nnz:", test_features.nnz)
print("Feature names unique: True")


## 7. 공식 실행 전 저장소 상태 확인

공식 runner는 source commit과 resolved config를 기록하므로 구현·config·노트북을 먼저 커밋해야 합니다. 노트북 실행 출력이나 autosave도 tracked notebook을 변경할 수 있으므로 상태를 확인합니다.


In [ ]:
status = run_command("git", "status", "--porcelain")
source_commit = run_command("git", "rev-parse", "HEAD")
print("Source commit:", source_commit)
if status:
    print("공식 실행 전 정리·커밋해야 할 변경:")
    print(status)
else:
    print("Clean worktree 확인")


## 8. 공식 EXP-156 canonical 5-fold 실행 명령

공식 실행은 노트북 autosave로 인한 dirty worktree를 피하기 위해 터미널에서 수행합니다. test는 각 fold 모델의 최종 확률 예측에만 사용되고, 특징 규칙·모델 선택에는 사용되지 않습니다. 아래 셀은 실행하지 않고 터미널에 붙여 넣을 명령만 표시합니다.


In [ ]:
commands = [
    "cd ~/git/open_cancer",
    ("jupyter nbconvert --ClearOutputPreprocessor.enabled=True --inplace "
     "notebooks/exp156_gene_variant_effect_compression.ipynb"),
    ("git add configs/exp156_gene_variant_effect_compression.yaml "
     "scripts/run_exp123_sparse_logistic_v1.py "
     "scripts/run_exp156_gene_variant_effect_compression.py "
     "src/open_cancer/compact_effect_features.py "
     "tests/test_compact_effect_features.py "
     "notebooks/exp156_gene_variant_effect_compression.ipynb"),
    'git commit -m "exp(#156): add compressed gene mutation effects"',
    "git status --short",
    "uv run python scripts/run_exp156_gene_variant_effect_compression.py",
]
print("\n\n".join(commands))


## 9. 실제 결과 확인

측정된 파일만 읽으며 점수나 재현 상태를 추정하지 않습니다.


In [ ]:
SLUG = "exp156_gene_variant_effect_compression"
METRICS_PATH = ROOT / "reports" / SLUG / "metrics.json"
COMPARISON_PATH = ROOT / "reproducibility" / SLUG / "comparison.json"
SUBMISSION_PATH = ROOT / "submissions" / f"{SLUG}.csv"

for path in (METRICS_PATH, COMPARISON_PATH, SUBMISSION_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)

metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
comparison = json.loads(COMPARISON_PATH.read_text(encoding="utf-8"))

display({
    "experiment_id": metrics["experiment_id"],
    "status": metrics["status"],
    "parent_experiment": metrics["parent_experiment"],
    "fold_macro_f1": [row["macro_f1"] for row in metrics["folds"]],
    "oof_macro_f1": metrics["oof"]["macro_f1"],
    "fold_mean": metrics["oof"]["fold_mean"],
    "fold_std": metrics["oof"]["fold_std"],
    "accuracy": metrics["oof"]["accuracy"],
    "log_loss": metrics["oof"]["log_loss"],
    "reproduction_passed": comparison["passed"],
    "submission": str(SUBMISSION_PATH.relative_to(ROOT)),
})


## 10. 전체 저장소 검증

실제 결과로 보고서와 History를 갱신한 뒤 다음 검증을 통과해야 합니다. 모델·OOF·test probability·가공 캐시는 Git에 추가하지 않습니다.


In [ ]:
validation = subprocess.run(
    ["uv", "run", "python", "scripts/validate_experiment.py"],
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(validation.stdout)
if validation.stderr:
    print(validation.stderr)
validation.check_returncode()
